# 3.3 Human-in-the-Loop (adaptado a lab local)

**Qué se ve aquí:** `HumanInTheLoopMiddleware` pausa el agente **antes** de ejecutar ciertas tools (`interrupt_on`). La respuesta trae `__interrupt__` y se reanuda con `Command(resume={"decisions": [...]})` usando **el mismo `thread_id`** (por eso hace falta un checkpointer). Decisiones: `approve`, `reject` (con mensaje) y `edit` (cambiando los argumentos).

> **Nota local:** cada decisión consume el interrupt pendiente. Para probar *reject* y *edit*, vuelve a correr la celda de `agent.invoke(...)` inicial antes de cada una (si no, no hay nada que reanudar).

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

### 📚 Teoría base del módulo 3: el ciclo del agente y dónde se engancha el middleware
`create_agent` es un **bucle**: el modelo piensa → (si pide tools) se ejecutan → el modelo vuelve a pensar → … hasta que responde sin pedir tools.

```
invoke ──► [before_agent] ──► ┌─► [before_model] ─► wrap_model_call( MODELO ) ─► [after_model] ─┐
                              │                                                                  │
                              │   ¿pidió tools? ── no ──► respuesta final ──► [after_agent] ──► fin
                              │        │ sí
                              └── wrap_tool_call( TOOL ) ◄┘
```
| Hook | Corre… | Ejemplo en este módulo |
|---|---|---|
| `before_agent` | 1 vez al empezar el `invoke` | borrar `ToolMessage`s (3.2) |
| `before_model` | antes de **cada** llamada al modelo | `SummarizationMiddleware` (3.2) |
| `wrap_model_call` | **envolviendo** cada llamada al modelo | cambiar modelo (3.4), tools (3.4, 3.5), prompt (`@dynamic_prompt` está hecho sobre esto) |
| `after_model` | después de cada respuesta del modelo | `HumanInTheLoopMiddleware` revisa los `tool_calls` y pausa (3.3) |
| `wrap_tool_call` | envolviendo cada ejecución de tool | reintentos o errores (como el interceptor MCP de M2.4) |

**Analogía:** el agente es un empleado; el middleware son los **supervisores** que revisan su trabajo en momentos fijos (al entrar, antes de hablar con el cliente, antes de enviar un correo…). El empleado no cambia; cambian las reglas que lo rodean.

### 📚 Teoría: Human-in-the-Loop (HITL)
**Idea:** hay acciones que un agente **no debe** ejecutar solo: enviar un correo, pagar, borrar datos. HITL pone un **semáforo** antes de esas tools: el agente se detiene y espera a que un humano decida.

**Cómo funciona por dentro:**
1. El modelo responde con un `AIMessage` que trae `tool_calls` (por ejemplo, `send_email`).
2. En `after_model`, el middleware revisa esos `tool_calls`. Si alguno está en `interrupt_on`, llama a **`interrupt(...)`** de LangGraph.
3. El grafo se **congela** y guarda todo en el **checkpointer** (por eso es obligatorio). `invoke` devuelve el state con la clave **`__interrupt__`**.
4. El humano revisa y responde con **`Command(resume={"decisions": [...]})`** usando el **mismo `thread_id`**. El grafo sigue exactamente donde quedó.

**Las 4 decisiones (LangChain 1.4):**
| Decisión | Qué pasa | Ejemplo |
|---|---|---|
| `approve` | La tool se ejecuta tal cual | `{"type": "approve"}` |
| `edit` | Se ejecuta con **otros argumentos** | `{"type": "edit", "edited_action": {"name": ..., "args": {...}}}` |
| `reject` | **No** se ejecuta; el modelo recibe un `ToolMessage` (status `error`) con tu motivo y puede intentar otra cosa | `{"type": "reject", "message": "..."}` |
| `respond` | **No** se ejecuta; tu texto se usa **como si fuera** el resultado de la tool | `{"type": "respond", "message": "..."}` (para tools tipo "pregúntale al usuario") |

**Configuración de `interrupt_on`:**
- `"send_email": True` → pausa y permite todas las decisiones.
- `"read_email": False` → nunca pausa (se ejecuta solo).
- Forma detallada: `{"send_email": {"allowed_decisions": ["approve", "reject"]}}` para limitar lo que el humano puede hacer.
- Si el modelo pide **varias** tools a la vez, llega **una decisión por cada** `action_request`, en el mismo orden.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [52]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

### 📘 Las tools de este ejemplo
- `read_email` **lee del state** (`runtime.state["email"]`): el correo no viene de una API, se lo pasamos al invocar. Es el mismo patrón de M2.2b.
- `send_email` es **falsa**: solo devuelve `"Email sent"`. En la vida real sería la acción peligrosa; por eso es la que lleva el semáforo.

In [53]:
# Sin cambios
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

### 📘 El agente: `state_schema` + `checkpointer` + HITL
- `EmailState(AgentState)` agrega el campo `email` al state.
- `checkpointer=InMemorySaver()` es **obligatorio**: sin la libreta, el agente no puede pausar y reanudar.
- `interrupt_on`: `read_email` libre, `send_email` con aprobación.
- `description_prefix`: el texto que verá el humano en la solicitud de aprobación.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent, AgentState
# from langgraph.checkpoint.memory import InMemorySaver
# from langchain.agents.middleware import HumanInTheLoopMiddleware
#
# class EmailState(AgentState):
#     email: str
#
# agent = create_agent(
#     model="gpt-5-nano",
#     tools=[read_email, send_email],
#     state_schema=EmailState,
#     checkpointer=InMemorySaver(),
#     middleware=[
#         HumanInTheLoopMiddleware(
#             interrupt_on={
#                 "read_email": False,
#                 "send_email": True,
#             },
#             description_prefix="Tool execution requires approval",
#         ),
#     ],
# )

In [54]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago.
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model=MODEL,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

### 📘 Qué esperar: el agente se detiene
El modelo llama a `read_email` (sin pausa) y luego pide `send_email` → **pausa**. La respuesta no termina en una respuesta final: trae `__interrupt__`.
> Guarda `config` (con el `thread_id`): es la "llave" para reanudar esta misma conversación.

### ⚠️ 1er intento (agente sin system prompt): no hubo pausa
Con el agente del curso, gemma4 **no llamó ninguna tool**. Respondió: *"I need to read your email first… Could you please tell me the email address you want me to read from?"* → sin `tool_calls` → **sin `__interrupt__`** (`KeyError`).

**Causa:** la docstring de `read_email` dice *"Read an email from the **given address**"*, pero la tool **no tiene argumentos** (el correo está en el state). El modelo chico creyó que le faltaba un dato y preguntó.
**Lección:** con HITL, si el modelo **no pide** la tool, el semáforo **nunca se activa**. Hay que verificar que el interrupt ocurrió (`"__interrupt__" in response`).
**Corrección:** system prompt explícito (celda siguiente).

In [55]:
# 🟢 ADAPTADO LOCAL — 2º intento: system prompt explícito
# Motivo: gemma4 no llamó read_email; la docstring dice "from the given address" (la tool no tiene
#         argumentos, el correo está en el state) y el modelo pidió la dirección en vez de actuar.
#         Línea 3 (agregada después): tras un Reject, gemma4 se rendía; gpt-5-nano reintentaba solo.
agent = create_agent(
    model=MODEL,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    system_prompt="""You are an email assistant.
    1. Call read_email (it takes NO arguments; the email is already loaded).
    2. Then call send_email with a short, polite reply in the body.
    3. If send_email is rejected, rewrite the email following the user's feedback and call send_email again.
    Never ask the user for addresses or confirmation.""",
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [56]:
# 🟢 ADAPTADO LOCAL — 2º intento: invocar el agente nuevo y verificar la pausa
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

for m in response["messages"]:
    print(f"{m.type:<6}", "tool_calls:" if getattr(m, "tool_calls", None) else "", 
          [tc["name"] for tc in getattr(m, "tool_calls", []) or []], "|", str(m.content)[:80])

if "__interrupt__" in response:
    print("\n✅ PAUSA (HITL). Borrador del correo:")
    print(response["__interrupt__"][0].value["action_requests"][0]["args"]["body"])
else:
    print("\n❌ Sin pausa: el modelo no pidió send_email")

human   [] | Please read my email and send a response immediately. Send the reply now in the 
ai     tool_calls: ['read_email'] | 
tool    [] | Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best,
ai     tool_calls: ['send_email'] | 

✅ PAUSA (HITL). Borrador del correo:
Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.


In [57]:
# Sin cambios
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'Thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          "Let's "
                                                                          'find '
                                                                          'another '
                                                                          'time '
                                                                          'that '
          

### 🧒 Cómo leer la salida de `pprint(response)` sin perderse
`response` es un **diccionario** (dict) con 3 llaves. `pprint` lo imprime entero, y como corta a 80 caracteres, **parte los textos largos palabra por palabra** (`'Hi ' 'John, ' 'No '…`): es el mismo texto, solo cortado.

```
response = {
  '__interrupt__': [ Interrupt(...) ],   ← solo si el agente está en PAUSA (qué quiere hacer y qué puedes decidir)
  'email':         "Hi Seán, ...",       ← el campo extra del state (EmailState)
  'messages':      [ ...lista... ]       ← la conversación, en orden
}
```
Dentro de `messages`, cada elemento es un mensaje. Lo que importa de cada uno:
| Tipo | Lo importante | Lo que puedes ignorar |
|---|---|---|
| `HumanMessage` | `content` (lo que pediste) | `id`, `additional_kwargs` |
| `AIMessage` | `content` (texto) y **`tool_calls`** (qué tool pide y con qué `args`) | `response_metadata` (tiempos de Ollama), `usage_metadata` (tokens) |
| `ToolMessage` | `content` (resultado de la tool), `name`, `status='error'` si falló o se rechazó | `id`, `tool_call_id` (une la respuesta con su pedido) |

> Truco: `AIMessage(content='', ... tool_calls=[...])` = el modelo **no habló**, **pidió una tool**. Es normal.

La celda siguiente define `ver(response)`: imprime lo mismo en **una línea por mensaje**. Úsala en lugar de `pprint` cuando quieras entender el flujo.

**Otra opción ya incluida en LangChain:** cada mensaje tiene `.pretty_print()`, que imprime separadores por tipo:
```python
for m in response["messages"]:
    m.pretty_print()
```

In [58]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================
Tool Calls:
  read_email (49f94923-474e-444e-a4eb-6fe0d8373b01)
 Call ID: 49f94923-474e-444e-a4eb-6fe0d8373b01
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.
================================== Ai Message ==================================
Tool Calls:
  send_email (54ffdd8b-a003-4bcd-9c03-592217a44546)
 Call ID: 54ffdd8b-a003-4bcd-9c03-592217a44546
  Args:
    body: Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.


In [59]:
# 🟢 EXTRA (aprendizaje): ver la respuesta con colores (una línea por mensaje)
AZUL, VERDE, AMAR, ROJO, MAG, GRIS, FIN = "\033[94m", "\033[92m", "\033[93m", "\033[91m", "\033[95m", "\033[90m", "\033[0m"

def ver(response):
    for i, m in enumerate(response["messages"]):
        if m.type == "human":
            print(f"{AZUL}[{i}] 👤 HUMANO:{FIN} {m.content}")
        elif m.type == "ai":
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f"{VERDE}[{i}] 🤖 MODELO pide tool →{FIN} {tc['name']}{GRIS}{tc['args']}{FIN}")
            elif m.content:
                print(f"{VERDE}[{i}] 🤖 MODELO responde:{FIN} {m.content}")
            else:
                print(f"{GRIS}[{i}] 🤖 MODELO: (respuesta vacía){FIN}")
        elif m.type == "tool":
            rechazada = getattr(m, "status", None) == "error"
            color, etiqueta = (ROJO, "⛔ RECHAZADA") if rechazada else (AMAR, "🔧 TOOL")
            print(f"{color}[{i}] {etiqueta} {m.name}:{FIN} {m.content}")
    if "__interrupt__" in response:
        valor = response["__interrupt__"][0].value
        acc = valor["action_requests"][0]
        print(f"\n{MAG}⏸️  PAUSA: quiere {acc['name']} con {acc['args']}{FIN}")
        print(f"{MAG}   decisiones permitidas: {valor['review_configs'][0]['allowed_decisions']}{FIN}")
    else:
        print(f"\n{GRIS}▶️  Sin pausa (terminó o no pidió una tool con semáforo){FIN}")
    print(f"{GRIS}state extra → email: {str(response.get('email', ''))[:60]}…{FIN}")

ver(response)

[0] 👤 HUMANO: Please read my email and send a response immediately. Send the reply now in the same thread.
[1] 🤖 MODELO pide tool → read_email{}
[2] 🔧 TOOL read_email: Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.
[3] 🤖 MODELO pide tool → send_email{'body': "Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán."}

⏸️  PAUSA: quiere send_email con {'body': "Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán."}
   decisiones permitidas: ['approve', 'edit', 'reject', 'respond']
state extra → email: Hi Seán, I'm going to be late for our meeting tomorrow. Can …


In [60]:
# Sin cambios
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán."}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John, Thanks for letting me know. Let\'s find another time that works for both of us. Best, Seán."}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='d82e415d7dd37aade2d1272187775f11')]


### 🧒 Cómo leer `response['__interrupt__'][0].value['action_requests'][0]['args']['body']`
Es como abrir **cajas dentro de cajas** (diccionarios `[...]` con nombre y listas `[0]` con posición):
| Paso | Qué abres | Qué hay |
|---|---|---|
| `response['__interrupt__']` | la caja de pausas | una **lista** de interrupciones |
| `[0]` | la primera pausa | un objeto `Interrupt` |
| `.value` | su contenido | un dict con `action_requests` y `review_configs` |
| `['action_requests']` | lo que el agente quiere hacer | una **lista** de tool calls pendientes |
| `[0]` | la primera acción | `{"name": "send_email", "args": {...}, "description": ...}` |
| `['args']['body']` | los argumentos → el cuerpo | **el texto del correo** que quiere enviar |

In [61]:
# Sin cambios
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.


### 🟢 Helper: crear una pausa nueva para cada decisión
Cada decisión **consume** la pausa. Si corres *Reject* después de *Approve* sobre el mismo thread, no hay nada que reanudar y el `Command` no hace nada (pasó en la 1ª corrida: Reject y Edit devolvieron el mismo historial con `'Email sent'`).
Esta función hace un `invoke` en un **thread nuevo** y deja la pausa lista. Cambia `config` para que las celdas del curso reanuden **ese** thread.

In [62]:
# 🟢 ADAPTADO LOCAL — helper: invocar en un thread nuevo y verificar la pausa
def nueva_pausa(thread_id: str):
    cfg = {"configurable": {"thread_id": thread_id}}
    r = agent.invoke(
        {
            "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
            "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
        },
        config=cfg,
    )
    if "__interrupt__" in r:
        print(f"✅ Pausa lista en thread '{thread_id}'. Borrador:")
        print("  ", r["__interrupt__"][0].value["action_requests"][0]["args"]["body"])
    else:
        print(f"❌ Thread '{thread_id}': el modelo no pidió send_email →", r["messages"][-1].content[:150])
    return cfg, r

## Approve

### 📘 Approve
`Command(resume=...)` no es un mensaje nuevo: es la **respuesta a la pausa**. Con `approve` la tool se ejecuta y el agente termina su turno.
> ⚠️ Después de esto ya no hay pausa pendiente. Para probar *reject*, vuelve a correr la celda del `invoke` inicial.

In [63]:
# Sin cambios
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='36e97819-1400-42bd-8851-fadec129073d'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T21:06:45.2554565Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3991271500, 'load_duration': 45168700, 'prompt_eval_count': 196, 'prompt_eval_duration': 3558378000, 'eval_count': 9, 'eval_duration': 363416000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10de3-e356-7f00-8f6f-ea88faa6f861-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '49f94923-474e-444e-a4eb-6fe0d8373b01', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 196, 'output_tok

In [64]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================
Tool Calls:
  read_email (49f94923-474e-444e-a4eb-6fe0d8373b01)
 Call ID: 49f94923-474e-444e-a4eb-6fe0d8373b01
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.
================================== Ai Message ==================================
Tool Calls:
  send_email (54ffdd8b-a003-4bcd-9c03-592217a44546)
 Call ID: 54ffdd8b-a003-4bcd-9c03-592217a44546
  Args:
    body: Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.
================================= Tool Message =================================
Name: send_email

Email sent
===================

## Reject

### 📘 Reject
El correo **no** se envía. El modelo recibe un `ToolMessage` con **`status='error'`** y tu motivo. Puede redactar otro borrador (→ **nueva pausa**) **o rendirse**.
> **1ª corrida (gemma4, sin la línea 3 del prompt):** se rindió: cerró con un `AIMessage` vacío, `¿Nueva pausa? False` y la celda del `['body']` dio `KeyError: '__interrupt__'`. No es un error de código: sin pausa no existe esa llave.
> **Corrección:** línea 3 del system prompt (*"If send_email is rejected, rewrite the email following the user's feedback and call send_email again"*). `gpt-5-nano` lo hacía solo; un modelo chico necesita que se lo digan.

In [65]:
# 🟢 ADAPTADO LOCAL — pausa nueva para probar Reject
config, response = nueva_pausa("reject")

✅ Pausa lista en thread 'reject'. Borrador:
   Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.


In [66]:
# Sin cambios
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'Thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          "Let's "
                                                                          'find '
                                                                          'another '
                                                                          'time '
                                                                          'that '
          

In [67]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================
Tool Calls:
  read_email (7eeebb9b-e209-4977-b7f8-64a02c74d584)
 Call ID: 7eeebb9b-e209-4977-b7f8-64a02c74d584
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.
================================== Ai Message ==================================
Tool Calls:
  send_email (1fd2be6b-e35f-4b12-b7ee-fd83302d600a)
 Call ID: 1fd2be6b-e35f-4b12-b7ee-fd83302d600a
  Args:
    body: Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.
================================= Tool Message =================================
Name: send_email

No please sign off - Your merc

In [68]:
# 🟢 ADAPTADO LOCAL — qué pasó después de Reject
ver(response)
print("\n¿Nueva pausa?", "__interrupt__" in response)

[0] 👤 HUMANO: Please read my email and send a response immediately. Send the reply now in the same thread.
[1] 🤖 MODELO pide tool → read_email{}
[2] 🔧 TOOL read_email: Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.
[3] 🤖 MODELO pide tool → send_email{'body': "Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán."}
[4] ⛔ RECHAZADA send_email: No please sign off - Your merciful leader, Seán.
[5] 🤖 MODELO pide tool → send_email{'body': "Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán."}

⏸️  PAUSA: quiere send_email con {'body': "Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán."}
   decisiones permitidas: ['approve', 'edit', 'reject', 'respond']
state extra → email: Hi Seán, I'm going to be late for our meeting tomorrow. Can …

¿Nueva pausa? True


In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: si el modelo no reintenta tras el rechazo, no hay pausa y da KeyError: '__interrupt__'.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

In [69]:
# 🟢 ADAPTADO LOCAL — leer el borrador nuevo solo si hubo nueva pausa
if "__interrupt__" in response:
    print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])
else:
    print("Sin nueva pausa: el modelo no reintentó send_email después del rechazo.")

Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.


## Edit

### 📘 Edit
Tú reescribes los argumentos (`edited_action`) y la tool se ejecuta **con tu versión**, no con la del modelo. Útil cuando el borrador está casi bien.
> **En mi corrida:** el `AIMessage` del historial quedó **con mis argumentos** (`"This is the last straw, you're fired!"`) y la tool devolvió `Email sent`. Ojo: depende de la versión. En LangChain 1.4.3 el historial conserva la llamada original del modelo y el cambio se aplica al ejecutar la tool. El concepto es el mismo: **se ejecuta lo que el humano editó**.

In [49]:
# 🟢 ADAPTADO LOCAL — pausa nueva para probar Edit
config, response = nueva_pausa("edit")

✅ Pausa lista en thread 'edit'. Borrador:
   Hi John, No problem at all. Let me know what time works best for you to reschedule. Best, Seán.


In [70]:
# Sin cambios
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='1f11d076-0750-45fc-bc6d-cf98810e849f'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T21:09:46.1250473Z', 'done': True, 'done_reason': 'stop', 'total_duration': 657462100, 'load_duration': 5017900, 'prompt_eval_count': 196, 'prompt_eval_duration': 288825000, 'eval_count': 9, 'eval_duration': 360036000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10de6-b2ba-7ae0-8d2f-b31ce3ecf59c-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '7eeebb9b-e209-4977-b7f8-64a02c74d584', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 196, 'output_tokens

In [71]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================
Tool Calls:
  read_email (7eeebb9b-e209-4977-b7f8-64a02c74d584)
 Call ID: 7eeebb9b-e209-4977-b7f8-64a02c74d584
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.
================================== Ai Message ==================================
Tool Calls:
  send_email (1fd2be6b-e35f-4b12-b7ee-fd83302d600a)
 Call ID: 1fd2be6b-e35f-4b12-b7ee-fd83302d600a
  Args:
    body: Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.
================================= Tool Message =================================
Name: send_email

No please sign off - Your merc

In [72]:
# 🟢 ADAPTADO LOCAL — qué pasó después de Edit
ver(response)
print("\n¿Nueva pausa?", "__interrupt__" in response)

[0] 👤 HUMANO: Please read my email and send a response immediately. Send the reply now in the same thread.
[1] 🤖 MODELO pide tool → read_email{}
[2] 🔧 TOOL read_email: Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.
[3] 🤖 MODELO pide tool → send_email{'body': "Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán."}
[4] ⛔ RECHAZADA send_email: No please sign off - Your merciful leader, Seán.
[5] 🤖 MODELO pide tool → send_email{'body': "This is the last straw, you're fired!"}
[6] 🔧 TOOL send_email: Email sent
[7] 🤖 MODELO: (respuesta vacía)

▶️  Sin pausa (terminó o no pidió una tool con semáforo)
state extra → email: Hi Seán, I'm going to be late for our meeting tomorrow. Can …

¿Nueva pausa? False


## 💡 Idea propia: las decisiones del humano como datos para entrenar
Cada decisión HITL es una **etiqueta gratis** hecha por un experto:
| Decisión | Qué dato produce | Para qué técnica |
|---|---|---|
| `approve` | (entrada, borrador) = **ejemplo bueno** | SFT (*supervised fine-tuning*) |
| `edit` | (entrada, borrador del modelo → **versión corregida**) | SFT con la corrección como respuesta ideal; par preferido/rechazado |
| `reject` + motivo | (entrada, borrador **malo**, crítica) | Pares de preferencia (DPO/RLHF) y casos de evaluación |
| `respond` | (pregunta del agente → respuesta humana) | Ejemplos para tools tipo "pregunta al usuario" |

Así se cierra el ciclo **producción → feedback humano → dataset → evaluación / fine-tuning → mejor modelo**. En LangSmith esto se hace con *feedback*, *annotation queues* y *datasets* (cursos 03 y 04).

**Cuidado:** los correos reales tienen datos personales (anonimizar), hace falta volumen y calidad, y conviene separar un conjunto de evaluación antes de entrenar.

La celda siguiente es un ejemplo mínimo: `decidir()` **registra** el borrador del modelo y tu decisión **antes** de reanudar (después ya no se puede: en algunas versiones, Edit reescribe el historial).

In [73]:
# 🟢 EXTRA (idea propia): registrar cada decisión HITL como un ejemplo de dataset
import json
from langgraph.types import Command

dataset = []

def decidir(config, response, decision):
    """Guarda (correo, borrador del modelo, decisión humana) y luego reanuda el agente."""
    borrador = response["__interrupt__"][0].value["action_requests"][0]["args"]
    r = agent.invoke(Command(resume={"decisions": [decision]}), config=config)
    dataset.append({
        "entrada": r.get("email"),
        "borrador_modelo": borrador.get("body"),
        "decision": decision["type"],
        "correccion_humana": decision.get("edited_action", {}).get("args", {}).get("body"),
        "motivo_rechazo": decision.get("message"),
    })
    return r

# Tres casos: aprobar, editar y rechazar (cada uno con su propia pausa)
cfg, resp = nueva_pausa("ds-approve"); decidir(cfg, resp, {"type": "approve"})
cfg, resp = nueva_pausa("ds-edit");    decidir(cfg, resp, {"type": "edit", "edited_action": {"name": "send_email", "args": {"body": "Hi John, sure. Does Thursday at 10am work? Best, Seán."}}})
cfg, resp = nueva_pausa("ds-reject");  decidir(cfg, resp, {"type": "reject", "message": "Too informal. Propose two concrete time slots."})

print("\nDataset (formato JSONL, una línea por ejemplo):")
for fila in dataset:
    print(json.dumps(fila, ensure_ascii=False))

✅ Pausa lista en thread 'ds-approve'. Borrador:
   Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.
✅ Pausa lista en thread 'ds-edit'. Borrador:
   Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.
✅ Pausa lista en thread 'ds-reject'. Borrador:
   Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.

Dataset (formato JSONL, una línea por ejemplo):
{"entrada": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.", "borrador_modelo": "Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.", "decision": "approve", "correccion_humana": null, "motivo_rechazo": null}
{"entrada": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.", "borrador_modelo": "Hi John, Thanks for letting me know. Let's find another time that works for both o

In [74]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Please read my email and send a response immediately. Send the reply now in the same thread.
================================== Ai Message ==================================
Tool Calls:
  read_email (7eeebb9b-e209-4977-b7f8-64a02c74d584)
 Call ID: 7eeebb9b-e209-4977-b7f8-64a02c74d584
  Args:
================================= Tool Message =================================
Name: read_email

Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.
================================== Ai Message ==================================
Tool Calls:
  send_email (1fd2be6b-e35f-4b12-b7ee-fd83302d600a)
 Call ID: 1fd2be6b-e35f-4b12-b7ee-fd83302d600a
  Args:
    body: Hi John, Thanks for letting me know. Let's find another time that works for both of us. Best, Seán.
================================= Tool Message =================================
Name: send_email

No please sign off - Your merc